# Estudio 06 · Fase 0 · Diagnóstico y techo de la V2

Solo lectura: no se entrena ninguna variante ni se escribe en `data/`. Se usan las predicciones fuera de muestra
guardadas de la V2 (`F_A3` del estudio 05: 5 semillas, h = 2, 3 y 4, bloques 2021–2025) y el protocolo congelado
(`denguekay.evaluation.protocol.evaluate`: Platt cronológico, p ≥ 0,50).

**Estados de la semana objetivo** (retrospectivos, por distrito, desde la etiqueta de gold):

- *inicio limpio*: semana brote entre las 2 primeras de un episodio sin brote en las 8 semanas previas;
- *en curso*: el resto de las semanas brote;
- *bajada*: semana sin brote hasta 8 semanas después de la última semana brote del distrito;
- *valle*: el resto de las semanas sin brote.

**Historia 2000–2025** (decisión de Rosa del 02/11/2026): un distrito sin registro en una semana de
`epi_piura_semanal.csv` tuvo 0 casos. La regla de brote es la de gold (media + 1,5 DE de la misma semana en los
5 años previos, ≥ 2 casos), así que la etiqueta existe desde 2005.

Todas las cifras son la mediana de las 5 semillas; el rango entre semillas está en el JSON.

In [1]:
import sys
import hashlib
import json

import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase0_diagnosis as d

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
METRICS = STUDY / "docs" / "metricas" / "fase0.json"
FIGURES = STUDY / "docs" / "figuras" / "fase0"

r = d.run_phase0()

## 1 · Reproducción de la V2 y HU0007-4

Debe dar las cifras del informe del estudio 05 (F1 2022 · h4 = 0,333; regresión que cumple en 2023 · h3 y h4).

In [2]:
hu = r["hu0007_4"]
hu[["horizonte", "bloque", "positivos", "precision_050", "recall_050", "f1_050", "razon_mae", "razon_rmse",
    "cumple_clasificacion", "cumple_regresion"]].round(3)

,horizonte,bloque,positivos,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_clasificacion,cumple_regresion
0,2,temporada_2022,431.0,0.734,0.687,0.709,0.919,0.938,False,False
1,2,temporada_2023,1039.0,0.899,0.821,0.858,0.818,0.869,True,False
2,2,temporada_2024,857.0,0.744,0.873,0.803,0.937,0.945,True,False
3,3,temporada_2022,431.0,0.835,0.462,0.594,0.883,0.922,False,False
4,3,temporada_2023,1039.0,0.897,0.768,0.828,0.747,0.818,False,True
5,3,temporada_2024,857.0,0.700,0.884,0.781,0.938,0.981,True,False
6,4,temporada_2022,431.0,0.960,0.202,0.333,0.838,0.905,False,False
7,4,temporada_2023,1039.0,0.911,0.733,0.811,0.708,0.779,False,True
8,4,temporada_2024,857.0,0.663,0.861,0.749,0.951,1.011,True,False


## 2 · Errores por estado de la semana objetivo

In [3]:
E = r["errores_por_estado"]
E[E.bloque.isin(d.HU_BLOCKS)][["horizonte", "bloque", "estado", "n", "positivos", "fn", "fp", "recall",
    "recall_persistencia", "fraccion_fn", "fraccion_fp", "razon_mae"]].round(3)

,horizonte,bloque,estado,n,positivos,fn,fp,recall,recall_persistencia,fraccion_fn,fraccion_fp,razon_mae
4,2,temporada_2022,bajada,514.0,0.0,0.0,99.0,NaN,NaN,0.000,0.935,0.816
5,2,temporada_2022,en_curso,382.0,382.0,91.0,0.0,0.762,0.764,0.674,0.000,1.043
6,2,temporada_2022,inicio_limpio,49.0,49.0,44.0,0.0,0.102,0.082,0.326,0.000,0.991
7,2,temporada_2022,valle,2435.0,0.0,0.0,7.0,NaN,NaN,0.000,0.065,0.835
8,2,temporada_2023,bajada,353.0,0.0,0.0,77.0,NaN,NaN,0.000,0.814,1.253
9,2,temporada_2023,en_curso,953.0,953.0,112.0,0.0,0.882,0.895,0.605,0.000,0.799
10,2,temporada_2023,inicio_limpio,86.0,86.0,73.0,0.0,0.151,0.081,0.395,0.000,0.885
11,2,temporada_2023,valle,1988.0,0.0,0.0,18.0,NaN,NaN,0.000,0.186,1.347
12,2,temporada_2024,bajada,809.0,0.0,0.0,233.0,NaN,NaN,0.000,0.904,0.985
13,2,temporada_2024,en_curso,823.0,823.0,87.0,0.0,0.894,0.834,0.798,0.000,0.913


## 3 · Calibración, semanas cerca del corte y techo diagnóstico de F1

Los techos usan un calibrador ajustado **con la misma temporada evaluada**: no son una evaluación válida ni deciden nada; solo dicen cuánto F1 a 0,50 podría recuperar una recalibración monótona perfecta.

In [4]:
C = r["calibracion"]
C[C.bloque.isin(d.HU_BLOCKS)][["horizonte", "bloque", "positivos", "prevalencia", "prob_media_calibrada",
    "brier", "brier_sin_calibrar", "ece", "ece_sin_calibrar", "brote_en_040_050", "no_brote_en_040_050",
    "f1_050", "f1_oraculo_platt", "f1_oraculo_isotonica", "f1_mejor_umbral", "fraccion_sin_calibrar_bajo_eps"]].round(3)

,horizonte,bloque,positivos,prevalencia,prob_media_calibrada,brier,brier_sin_calibrar,ece,ece_sin_calibrar,brote_en_040_050,no_brote_en_040_050,f1_050,f1_oraculo_platt,f1_oraculo_isotonica,f1_mejor_umbral,fraccion_sin_calibrar_bajo_eps
1,2,temporada_2022,431.0,0.128,0.145,0.054,0.053,0.027,0.021,39.0,68.0,0.709,0.701,0.703,0.716,0.032
2,2,temporada_2023,1039.0,0.307,0.282,0.063,0.065,0.033,0.049,57.0,76.0,0.858,0.861,0.861,0.864,0.015
3,2,temporada_2024,857.0,0.254,0.289,0.075,0.071,0.042,0.019,34.0,83.0,0.803,0.801,0.792,0.806,0.081
5,3,temporada_2022,431.0,0.128,0.126,0.059,0.056,0.039,0.028,87.0,59.0,0.594,0.700,0.702,0.717,0.026
6,3,temporada_2023,1039.0,0.307,0.267,0.075,0.080,0.046,0.075,81.0,79.0,0.828,0.840,0.842,0.843,0.013
7,3,temporada_2024,857.0,0.254,0.305,0.083,0.078,0.061,0.030,26.0,73.0,0.781,0.789,0.788,0.797,0.055
9,4,temporada_2022,431.0,0.128,0.109,0.068,0.062,0.046,0.034,93.0,26.0,0.333,0.649,0.680,0.696,0.025
10,4,temporada_2023,1039.0,0.307,0.256,0.084,0.093,0.061,0.096,80.0,78.0,0.811,0.828,0.826,0.830,0.012
11,4,temporada_2024,857.0,0.254,0.320,0.094,0.088,0.074,0.039,42.0,95.0,0.749,0.758,0.756,0.768,0.038


In [5]:
fig = d.reliability_figure(r["evaluaciones"][17]["datos"])
diag.save_figure(fig, FIGURES, "fiabilidad_v2_por_temporada")

WindowsPath('C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/research/06_model_improvement/docs/figuras/fase0/fiabilidad_v2_por_temporada.png')

## 4 · Persistencia: qué fracción de las semanas brote ya acierta

In [6]:
P = r["persistencia"]
P[P.bloque.isin(d.HU_BLOCKS)][["horizonte", "bloque", "positivos", "recall_persistencia", "recall_modelo",
    "ambos", "solo_persistencia", "solo_modelo", "ninguno", "recall_modelo_donde_persistencia_falla"]].round(3)

,horizonte,bloque,positivos,recall_persistencia,recall_modelo,ambos,solo_persistencia,solo_modelo,ninguno,recall_modelo_donde_persistencia_falla
1,2,temporada_2022,431.0,0.687,0.687,0.661,0.026,0.023,0.290,0.074
2,2,temporada_2023,1039.0,0.828,0.821,0.797,0.031,0.024,0.148,0.140
3,2,temporada_2024,857.0,0.805,0.873,0.804,0.001,0.069,0.126,0.353
5,3,temporada_2022,431.0,0.694,0.462,0.462,0.232,0.000,0.306,0.000
6,3,temporada_2023,1039.0,0.783,0.768,0.737,0.046,0.031,0.186,0.142
7,3,temporada_2024,857.0,0.784,0.884,0.784,0.000,0.102,0.114,0.470
9,4,temporada_2022,431.0,0.664,0.202,0.202,0.462,0.000,0.336,0.000
10,4,temporada_2023,1039.0,0.757,0.733,0.699,0.059,0.035,0.208,0.143
11,4,temporada_2024,857.0,0.740,0.861,0.735,0.005,0.127,0.133,0.489


## 5 · Historia previa a 2017

In [7]:
r["auditoria_historia"]

{'filas_silver_fuera_de_la_grilla': 3,
 'casos_silver_fuera_de_la_grilla': 4,
 'filas_solapadas_distintas': 0,
 'anios_solapados': [2017, 2024],
 'filas_etiqueta_comparadas_con_gold': 30485,
 'etiquetas_distintas_de_gold': 0,
 'umbral_diferencia_maxima': 0.0}

In [8]:
r["historia_anual"]

,anio,casos,semanas_brote,distritos_con_brote
0,2005,51,1,1
1,2006,865,28,8
2,2007,282,38,6
3,2008,1702,101,12
4,2009,4029,168,21
5,2010,8393,222,25
6,2011,183,21,4
7,2012,1181,85,12
8,2013,1979,60,9
9,2014,2675,103,10


In [9]:
r["historia_temporadas"]

,temporada,casos,semanas_brote,semanas,inicios_limpios,verificables,distritos
0,2005,37,0,34,0.0,0.0,0.0
1,2006,827,20,52,7.0,7.0,7.0
2,2007,310,43,52,10.0,10.0,7.0
3,2008,1432,85,52,11.0,11.0,10.0
4,2009,4243,174,53,21.0,21.0,19.0
5,2010,8448,234,52,30.0,30.0,25.0
6,2011,141,13,52,4.0,4.0,2.0
7,2012,1098,64,52,15.0,15.0,12.0
8,2013,2095,85,52,9.0,9.0,8.0
9,2014,2190,58,52,11.0,11.0,9.0


## 6 · Salidas: métricas y MLflow

In [10]:
def records(t):
    return diag.to_json(t)

content = {
    "fuente_predicciones": d.V2_PREDICTIONS.name,
    "definiciones": {"semanas_inicio": d.ONSET_WEEKS, "semanas_bajada": d.DECAY_WEEKS,
                     "sin_brote_previo": diag.NO_PREVIOUS_OUTBREAK, "corte_cercano": list(d.NEAR_CUTOFF),
                     "regla_brote": d.OUTBREAK_RULE, "historia_desde": d.HISTORY_START},
    "hu0007_4": records(hu), "errores_por_estado": records(E), "calibracion": records(C),
    "persistencia": records(P), "fiabilidad_semilla_17": records(r["evaluaciones"][17]["fiabilidad"]),
    "historia_anual": records(r["historia_anual"]), "historia_temporadas": records(r["historia_temporadas"]),
    "auditoria_historia": r["auditoria_historia"],
}
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
fingerprint = hashlib.sha256(METRICS.read_bytes()).hexdigest()
fingerprint[:12]

'605ab13a8861'

In [11]:
cell = lambda t, b, h: t[t.bloque.eq(b) & t.horizonte.eq(h)].iloc[0]
key = {f"{m}.{b[-4:]}.h{h}": float(cell(t, b, h)[c])
       for b in d.HU_BLOCKS for h in (2, 3, 4)
       for m, t, c in (("f1_050", C, "f1_050"), ("f1_oraculo_isotonica", C, "f1_oraculo_isotonica"),
                       ("recall_persistencia", P, "recall_persistencia"))}
tracking.log_study(
    "mejora_modelo_fase0_diagnostico", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "0", "predicciones": d.V2_PREDICTIONS.name, "semillas": "17,29,41,53,67",
                       "configuracion": "V2 (CONFIG_2026)"}, "metricas": key},
    artifacts=[METRICS, FIGURES / "fiabilidad_v2_por_temporada.png"],
    fingerprint=fingerprint, description="Fase 0 del estudio 06: diagnóstico de la V2 sin entrenar.")

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio aebe6121974545bb850172b82d51f634 en 'denguekay/mejora_modelo_fase0_diagnostico' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'aebe6121974545bb850172b82d51f634'